# Implement LoRA on a Linear Layer — Solution

**Difficulty**: 🟡 Medium

**Companies**: Meta, Google, Anthropic, OpenAI, Databricks

---

### Problem Statement

Low-Rank Adaptation (LoRA) is one of the most important parameter-efficient fine-tuning techniques in modern deep learning. Instead of fine-tuning all parameters of a pretrained model, LoRA freezes the original weights and injects small, trainable low-rank matrices into each layer. This dramatically reduces the number of trainable parameters while maintaining model quality.

The key idea: instead of learning a full update ΔW for a frozen weight W, LoRA constrains the update to rank r ≪ min(d_in, d_out). In the paper's column-vector notation, h = W₀x + ΔWx = W₀x + BAx with B ∈ ℝ^(d_out×r) and A ∈ ℝ^(r×d_in).

This notebook stores the two factors in **row-vector layout** instead, so they apply directly to a batch `x` of shape `(batch, d_in)`: `A` is `(d_in, r)` and `B` is `(r, d_out)`. The forward pass becomes:

```
output = linear(x) + (α/r) · (x @ A) @ B        # linear(x) = x @ Wᵀ + b, W is (d_out, d_in)
```

so the equivalent dense update to `nn.Linear.weight` is ΔW = (α/r) · (A @ B)ᵀ. Only A and B (r·(d_in + d_out) numbers per layer) are trained.

Your task is to implement LoRA from scratch, apply it to an existing model, and merge the adapted weights back.

---

### Requirements

1. **LoRALinear** — A module that wraps an existing `nn.Linear`, adds low-rank matrices A and B, and freezes the original weights.
2. **apply_lora** — A function that injects LoRA into specified layers of an existing model.
3. **merge_lora** — Fold each adapter back into its linear layer, W_new = W + (α/r) · (A @ B)ᵀ (shape `(d_out, d_in)`, the layout of `nn.Linear.weight`), and replace the `LoRALinear` with a plain `nn.Linear`, so inference costs nothing extra.
4. **Parameter Efficiency** — Show that trainable parameters are much fewer than total parameters.
5. **Training** — Train a simple model with LoRA on synthetic data.

---

### Constraints

- ✅ B is initialized to zeros and A randomly, so ΔW = 0 at step 0 (the wrapped model starts exactly at the pretrained function) while B still receives a gradient from the first step. The reference implementations (loralib, PEFT) draw A with `kaiming_uniform_(a=√5)` using fan_in = d_in, i.e. U(−1/√d_in, 1/√d_in); the paper describes a random Gaussian A. Careful: `nn.init` reads fan_in from dim 1, which for A stored as `(d_in, r)` is r, not d_in.
- ✅ Original weights must be frozen (requires_grad=False).
- ✅ Merged model must produce identical outputs to the LoRA model.
- ❌ Do **not** use any existing LoRA libraries (peft, loralib, etc.).

---

<details>
  <summary>💡 Hint</summary>

  - Initialize B with zeros and A with small random values (bound 1/√d_in). The LoRA contribution then starts at exactly zero. Why not make both zero? Then the gradients of both would be zero and the adapter could never move.
  - The scaling factor is `alpha / rank`. This controls the magnitude of the LoRA update relative to the original weights.
  - When merging, add `(alpha/r) * (A @ B).T` (equivalently `(alpha/r) * B.T @ A.T`) to the original weight: `A @ B` is `(d_in, d_out)` and `nn.Linear.weight` is `(d_out, d_in)`. Do it under `torch.no_grad()`.
  - To apply LoRA to a model, walk the children recursively (`named_children`) and `setattr` a `LoRALinear` wrapper over each matching `nn.Linear`.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy
import math

In [ ]:
# A simple pretrained model to apply LoRA to
class SimpleModel(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, output_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

torch.manual_seed(42)
model = SimpleModel()
print("Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
class LoRALinear(nn.Module):
    """
    Wraps an existing nn.Linear layer with low-rank adaptation.

    Forward: output = linear(x) + (alpha / r) * (x @ A) @ B
    where A is (d_in, r) and B is (r, d_out). The wrapped layer is frozen.
    """
    def __init__(self, linear: nn.Linear, rank: int = 4, alpha: float = 1.0):
        super().__init__()
        self.linear = linear
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        d_in = linear.in_features
        d_out = linear.out_features

        # Freeze the pretrained weight and bias
        for param in self.linear.parameters():
            param.requires_grad = False

        # A random, B zero: the adapter starts as an exact no-op (delta W = 0), and
        # because A is non-zero, B gets a gradient from the very first step.
        # Scale of A matches loralib/PEFT: kaiming_uniform_(a=sqrt(5)) with
        # fan_in = d_in, i.e. U(-1/sqrt(d_in), 1/sqrt(d_in)). We set the bound
        # explicitly because nn.init would read fan_in from dim 1, which is r here.
        factory = dict(device=linear.weight.device, dtype=linear.weight.dtype)
        bound = 1.0 / math.sqrt(d_in)
        self.A = nn.Parameter(torch.empty(d_in, rank, **factory).uniform_(-bound, bound))
        self.B = nn.Parameter(torch.zeros(rank, d_out, **factory))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass: original output + scaled low-rank adaptation.
        output = linear(x) + (alpha/r) * (x @ A) @ B
        """
        # (x @ A) first: two skinny matmuls, never a (d_in, d_out) matrix
        return self.linear(x) + (x @ self.A) @ self.B * self.scaling


def apply_lora(model: nn.Module, rank: int = 4, alpha: float = 1.0,
               target_modules: list = None) -> nn.Module:
    """
    Replace target nn.Linear layers in model with LoRALinear wrappers.
    A child is targeted when one of the target_modules strings occurs in its name.
    """
    if target_modules is None:
        target_modules = []

    for name, module in model.named_children():
        if isinstance(module, nn.Linear) and any(t in name for t in target_modules):
            setattr(model, name, LoRALinear(module, rank=rank, alpha=alpha))
        else:
            apply_lora(module, rank, alpha, target_modules)

    return model


def merge_lora(model: nn.Module) -> nn.Module:
    """
    Merge LoRA weights back into the original linear layers and drop the wrappers.
    W_new = W_old + (alpha/r) * (A @ B).T
    """
    for name, module in model.named_children():
        if isinstance(module, LoRALinear):
            with torch.no_grad():
                # A @ B is (d_in, d_out); nn.Linear stores weight as (d_out, d_in)
                module.linear.weight += module.scaling * (module.A @ module.B).T
            # The wrapped layer now carries the merged weight (same device/dtype)
            setattr(model, name, module.linear)
        else:
            merge_lora(module)

    return model


def count_parameters(model: nn.Module):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [ ]:
# === Validation ===
torch.manual_seed(42)

# Create model and apply LoRA
model = SimpleModel()
lora_model = copy.deepcopy(model)
apply_lora(lora_model, rank=4, alpha=1.0, target_modules=['fc1', 'fc2'])

# 1. Parameter count comparison
print("=== Parameter Count Test ===")
total_orig, train_orig = count_parameters(model)
total_lora, train_lora = count_parameters(lora_model)
print(f"Original model: {total_orig:,} total, {train_orig:,} trainable")
print(f"LoRA model:     {total_lora:,} total, {train_lora:,} trainable")
print(f"Trainable ratio: {train_lora/total_lora*100:.1f}%")
assert train_lora < train_orig, "LoRA should have fewer trainable params!"
print("PASSED\n")

# 2. Train with LoRA on synthetic data
print("=== Training Test ===")
X = torch.randn(200, 64)
y = torch.randint(0, 10, (200,))
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, lora_model.parameters()), lr=1e-3)

losses = []
for epoch in range(50):
    optimizer.zero_grad()
    logits = lora_model(X)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss:   {losses[-1]:.4f}")
assert losses[-1] < losses[0], "Loss should decrease during training!"
print("PASSED\n")

# 3. Merge and compare outputs
print("=== Merge Test ===")
test_input = torch.randn(10, 64)
lora_model.eval()
with torch.no_grad():
    lora_output = lora_model(test_input)

merged_model = copy.deepcopy(lora_model)
merge_lora(merged_model)
merged_model.eval()
with torch.no_grad():
    merged_output = merged_model(test_input)

print(f"Max difference: {(lora_output - merged_output).abs().max().item():.2e}")
assert torch.allclose(lora_output, merged_output, atol=1e-5), "Merged model output differs from LoRA model!"
print("PASSED\n")

# 4. Verify merged model has no LoRA modules
print("=== Clean Merge Test ===")
has_lora = any(isinstance(m, LoRALinear) for m in merged_model.modules())
assert not has_lora, "Merged model should not contain LoRALinear modules!"
print("No LoRALinear modules remain after merge.")
print("PASSED\n")

print("All tests passed!")